# MoodTune — Phase 5: Model Training and Selection

This notebook trains the required classifiers on pseudo-labels, performs a held-out stratified evaluation, tunes the strongest baseline, saves one pipeline, and visualizes comparison results. It must not be interpreted as validation of human emotional understanding.

In [ ]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

project_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((root for root in project_roots if (root / 'ml').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from the MoodTune project or a child directory.')
sys.path.insert(0, str(PROJECT_ROOT))
from ml.config.mood_config import RANDOM_SEED, SONG_MOODS
from ml.training.train_mood_classifier import (METRICS_PATH, MODEL_FEATURES, TARGET_COLUMN, load_training_data, train_and_select_model)


## Training and comparison

All candidate models use their own pipeline. The split is fixed, stratified, and performed after verifying unique track IDs, so a repeated song cannot appear on both sides of the evaluation.

In [ ]:
final_model, results = train_and_select_model()
comparison = pd.DataFrame({
    name: {metric: values[metric] for metric in ['accuracy', 'macro_f1', 'weighted_f1', 'fit_seconds']}
    for name, values in results['baselines'].items()
}).T.sort_values('macro_f1', ascending=False)
display(comparison)

fig, axis = plt.subplots(figsize=(9, 4))
comparison[['macro_f1', 'weighted_f1']].plot.bar(ax=axis)
axis.set(title='Baseline Model Comparison', xlabel='Model', ylabel='F1 score', ylim=(0.6, 1.0))
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()
print(json.dumps(results['final_model'], indent=2))

## Final held-out evaluation

The test partition was untouched during the three-fold tuning search. The confusion matrix identifies which pseudo-label boundaries remain difficult for the final pipeline.

In [ ]:
X, y = load_training_data()
_, X_test, _, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y)
predictions = final_model.predict(X_test)
display(pd.DataFrame(results['final_model']['test_metrics']['classification_report']).T)
figure, axis = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, labels=list(SONG_MOODS), xticks_rotation=45, cmap='Blues', ax=axis)
axis.set_title('Final Logistic Regression: Held-Out Confusion Matrix')
plt.tight_layout()
plt.show()
print(f'Metrics saved to: {METRICS_PATH}')